# Coding Exercise 6
## Poisson · MA(5) · VAR(3) · Ridge Regression · Regression Tree · Neural Network
> All implementations are generic — change parameters in the clearly-labelled
> **"Parameters"** block at the top of each question and the rest adapts automatically.


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats           # used ONLY for chi2.sf and chi2 distribution — NOT chisquare()
from sklearn.tree            import DecisionTreeRegressor, plot_tree, export_text
from sklearn.neural_network  import MLPRegressor
from sklearn.model_selection import KFold
from sklearn.metrics         import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)


---
## Q1 – Generate Poisson(λ) using `numpy.random.exponential` + Manual Chi-Square Test

### Key idea
Inter-arrival times of a Poisson(λ) process are i.i.d. Exp(λ) (mean = 1/λ).  
The number of arrivals in a unit interval [0, 1] is Poisson(λ).

**Vectorised generation:** pre-allocate a `(T × max_k)` matrix of Exp(λ) draws,  
take cumulative sums row-wise, count how many partial sums stay ≤ 1.

### Chi-square test (manual — no `scipy.stats.chisquare`)
$$\chi^2 = \sum_i \frac{(O_i - E_i)^2}{E_i}$$
Bins where $O_i > 0$ but $E_i \le 0.1$ are **removed** before computing the statistic.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS  (change freely)
# ══════════════════════════════════════════════════════════════════
LAM   = 1.72      # Poisson rate λ
T_Q1  = 10_000    # number of samples to generate
LOW_E = 0.1       # threshold: remove bin if O_i > 0 AND E_i ≤ LOW_E
# ══════════════════════════════════════════════════════════════════

# ── Step 1: Generate Poisson(LAM) via exponential inter-arrivals ───────────────
# For each of T_Q1 samples, accumulate Exp(1/LAM) draws until cumsum > 1;
# count how many steps were taken — that count is Poisson(LAM).
max_k  = int(LAM + 15 * np.sqrt(LAM)) + 20   # safe upper bound for columns
# Each entry ~ Exp(1/LAM)  ↔  mean = 1/LAM
E_mat  = np.random.exponential(1.0 / LAM, size=(T_Q1, max_k))
X_pois = (np.cumsum(E_mat, axis=1) <= 1.0).sum(axis=1)   # shape (T_Q1,)

print(f"Poisson(λ={LAM}) — n={T_Q1}")
print(f"  Empirical mean : {X_pois.mean():.4f}  (theory {LAM})")
print(f"  Empirical var  : {X_pois.var():.4f}  (theory {LAM})")

# ── Step 2: Observed and expected frequency vectors ────────────────────────────
max_val = int(X_pois.max())
ks      = np.arange(0, max_val + 1)
O_full  = np.array([np.sum(X_pois == k) for k in ks], dtype=float)
E_full  = np.array([stats.poisson.pmf(k, LAM) * T_Q1 for k in ks])

# ── Step 3: Remove bins where O_i > 0 AND E_i ≤ LOW_E ────────────────────────
bad_bins = (O_full > 0) & (E_full <= LOW_E)
O_clean  = O_full[~bad_bins]
E_clean  = E_full[~bad_bins]
ks_clean = ks[~bad_bins]

print(f"  Bins before removal : {len(O_full)}")
print(f"  Bins removed        : {bad_bins.sum()}  (O>0, E≤{LOW_E})")
print(f"  Bins after removal  : {len(O_clean)}")

print("  Observed frequency vector (after removal):")
print("  ", O_clean.astype(int))
print("  Expected frequency vector (after removal):")
print("  ", np.round(E_clean, 4))

# ── Step 4: Chi-square statistic and p-value ───────────────────────────────────
chi2_stat  = np.sum((O_clean - E_clean) ** 2 / E_clean)
df_chi2    = len(O_clean) - 1          # degrees of freedom
p_val_chi2 = stats.chi2.sf(chi2_stat, df_chi2)   # chi2 distribution — NOT chisquare()

print(f"  Chi-square statistic : {chi2_stat:.4f}")
print(f"  Degrees of freedom   : {df_chi2}")
print(f"  p-value              : {p_val_chi2:.4f}")
print(f"  Accept H₀ (p > 0.05)? {'✓ Yes' if p_val_chi2 > 0.05 else '✗ No'}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(ks, O_full / T_Q1, alpha=0.6, label='empirical PMF')
axes[0].plot(ks, E_full / T_Q1, 'r-o', ms=5, label=f'Poisson({LAM}) PMF')
axes[0].set_title(f'Poisson(λ={LAM}) — simulated vs theory')
axes[0].legend(); axes[0].set_xlabel('k')

axes[1].bar(ks_clean, O_clean, alpha=0.6, label='O_i (observed)')
axes[1].plot(ks_clean, E_clean, 'r-o', ms=5, label='E_i (expected)')
axes[1].set_title(f'After removal  χ²={chi2_stat:.2f}  p={p_val_chi2:.4f}')
axes[1].legend(); axes[1].set_xlabel('k')
plt.tight_layout(); plt.show()


---
## Q2 – MA(5) Random Process (numpy only)

### Model
$$X_t = \varepsilon_t + a_1\varepsilon_{t-1} + \cdots + a_5\varepsilon_{t-5}$$

$\{\varepsilon_t\}$ i.i.d. Uniform with mean 0.5 and variance 2.

### Theoretical moments (with $a_0=1$)
| Quantity | Formula |
|---|---|
| Mean | $\mu_X = \mu_\varepsilon\sum_{j=0}^5 a_j$ |
| Variance $\gamma(0)$ | $\sigma^2_\varepsilon\sum_{j=0}^5 a_j^2$ |
| $\gamma(s),\;1\le s\le 5$ | $\sigma^2_\varepsilon\sum_{j=0}^{5-s} a_j a_{j+s}$ |
| $\gamma(s),\;s > 5$ | $0$ |

**Uniform distribution** with mean $\mu$ and variance $\sigma^2$:
$U[\mu - \sqrt{3\sigma^2},\; \mu + \sqrt{3\sigma^2}]$.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS  (change freely — only numpy used)
# ══════════════════════════════════════════════════════════════════
T_Q2   = 10_000
q_ma   = 5                                         # MA order (change to any q)
a_coeffs = [0.6, -0.4, 0.3, -0.2, 0.5]           # a1 … a5  (any real numbers)
EPS_MEAN_Q2 = 0.5
EPS_VAR_Q2  = 2.0
S_MAX  = 7                                         # print γ(s) for s = 1..S_MAX
# ══════════════════════════════════════════════════════════════════

# ── Uniform noise with given mean and variance ─────────────────────────────────
# Uniform[a,b]: mean=(a+b)/2, var=(b-a)^2/12
# → half-width h = sqrt(3*var), a = mean-h, b = mean+h
h_unif   = np.sqrt(3 * EPS_VAR_Q2)
eps_low  = EPS_MEAN_Q2 - h_unif
eps_high = EPS_MEAN_Q2 + h_unif
eps_ma2  = np.random.uniform(eps_low, eps_high, T_Q2 + q_ma)  # (T+q,)

# Empirical check on noise
print(f"Noise Uniform[{eps_low:.4f}, {eps_high:.4f}]")
print(f"  Empirical mean = {eps_ma2.mean():.4f}  (theory {EPS_MEAN_Q2})")
print(f"  Empirical var  = {eps_ma2.var():.4f}   (theory {EPS_VAR_Q2})")

# ── Generate MA(q) process ─────────────────────────────────────────────────────
a = np.array([1.0] + list(a_coeffs))   # a[0]=1, a[1..q]=coefficients
# X[t] = sum_{j=0}^{q} a[j] * eps[t+q-j]  → eps[t+q], eps[t+q-1], …, eps[t]
X_ma2 = np.array([a @ eps_ma2[t: t + q_ma + 1][::-1] for t in range(T_Q2)])

# ── Theoretical moments ────────────────────────────────────────────────────────
mu_theory  = a.sum() * EPS_MEAN_Q2
acv_theory = np.zeros(S_MAX + 1)
for s in range(q_ma + 1):
    acv_theory[s] = EPS_VAR_Q2 * np.dot(a[:q_ma - s + 1], a[s:])
# acv_theory[s] = 0 for s > q_ma  (already 0)

# ── Empirical moments ──────────────────────────────────────────────────────────
mu_emp  = X_ma2.mean()
n       = len(X_ma2)
Xc      = X_ma2 - mu_emp
acv_emp = np.array([np.mean(Xc[:n-s] * Xc[s:]) for s in range(S_MAX + 1)])

# ── Print comparison ───────────────────────────────────────────────────────────
print(f"\nMA({q_ma})   a = {a_coeffs}")
print(f"{'Quantity':30}  {'Theory':>12}  {'Empirical':>12}")
print(f"{'Mean':30}  {mu_theory:>12.4f}  {mu_emp:>12.4f}")
print(f"{'Variance g(0)':30}  {acv_theory[0]:>12.4f}  {acv_emp[0]:>12.4f}")
for s in range(1, S_MAX + 1):
    print(f"{'g('+str(s)+')':30}  {acv_theory[s]:>12.4f}  {acv_emp[s]:>12.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(X_ma2[:400], lw=0.7); axes[0].set_title(f'MA({q_ma}) realisation (first 400 pts)')
lags_q2 = np.arange(S_MAX + 1)
axes[1].bar(lags_q2, acv_emp, alpha=0.6, label='empirical γ(s)')
axes[1].plot(lags_q2, acv_theory, 'r-o', ms=6, label='theory γ(s)')
axes[1].set_title('Autocovariance γ(s)'); axes[1].legend()
axes[2].hist(X_ma2, bins=50, density=True, alpha=0.6); axes[2].set_title('Histogram of X_t')
plt.tight_layout(); plt.show()


---
## Q3 – VAR(3) Random Process (numpy only)

### (a) Model
$$\mathbf{X}_t = A_1\mathbf{X}_{t-1} + A_2\mathbf{X}_{t-2} + A_3\mathbf{X}_{t-3} + \boldsymbol{\varepsilon}_t$$

Noise: $\boldsymbol{\varepsilon}_t \stackrel{iid}{\sim} \mathcal{N}([1,-1]^\top,\,[[3,2],[2,3]])$

### (b) Stationarity via eigenvalue method
Build the companion matrix:
$$C = \begin{bmatrix}A_1 & A_2 & A_3\\ I & 0 & 0\\ 0 & I & 0\end{bmatrix}$$
The process is **stationary** iff all eigenvalues of $C$ satisfy $|\lambda| < 1$.

### (c) Estimate $A_1, A_2, A_3$ from data by OLS
Stack $p$ lags into $Z$, then $\hat{B} = (Z^\top Z)^{-1}Z^\top Y$.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS  (change freely — only numpy used)
# ══════════════════════════════════════════════════════════════════
T_Q3 = 10_000
BURN = 500          # burn-in to reach stationarity

A1 = np.array([[ 0.3, -0.1],
               [-0.1,  0.3]])
A2 = np.array([[-0.3, -0.1],
               [-0.1, -0.3]])
A3 = np.array([[-0.3,  0.1],
               [ 0.1, -0.3]])

MU_EPS  = np.array([1.0, -1.0])             # noise mean vector
COV_EPS = np.array([[3.0, 2.0],             # noise covariance matrix
                     [2.0, 3.0]])
# ══════════════════════════════════════════════════════════════════

p_var = 3    # VAR order (must match number of A matrices)
d_var = A1.shape[0]
A_list = [A1, A2, A3]

# ── (a) Generate VAR(p) ────────────────────────────────────────────────────────
def generate_var_numpy(A_list, T, mu_eps, Sigma_eps, burn=500):
    """
    Generate VAR(p) using only numpy.
    A_list : [A1, …, Ap], each (d,d)
    Returns X of shape (T, d)
    """
    p = len(A_list)
    d = A_list[0].shape[0]
    Ttot = T + burn
    # Multivariate normal noise: Cholesky decomposition (numpy only)
    L = np.linalg.cholesky(Sigma_eps)         # Sigma = L L^T
    z = np.random.randn(Ttot, d)
    eps = (L @ z.T).T + mu_eps                 # shape (Ttot, d)

    X = np.zeros((Ttot, d))
    for t in range(p, Ttot):
        X[t] = sum(A_list[k] @ X[t - k - 1] for k in range(p)) + eps[t]
    return X[burn:], eps[burn:]


X_var3, eps_var3 = generate_var_numpy(A_list, T_Q3, MU_EPS, COV_EPS, BURN)

# Theoretical process mean: μ_X = (I - A1 - A2 - A3)^{-1} μ_ε
mu_X_theory = np.linalg.solve(np.eye(d_var) - sum(A_list), MU_EPS)

print("=== (a) VAR(3) Generation ===")
print(f"  Empirical process mean : {X_var3.mean(axis=0).round(4)}")
print(f"  Theoretical mean       : {mu_X_theory.round(4)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ci, ax in enumerate(axes):
    ax.plot(X_var3[:400, ci], lw=0.7)
    ax.set_title(f'VAR(3) — Component {ci+1} (first 400 pts)')
plt.tight_layout(); plt.show()

# ── (b) Stationarity via eigenvalue method ────────────────────────────────────
# Companion matrix C (d*p × d*p)
C = np.zeros((d_var * p_var, d_var * p_var))
for k in range(p_var):
    C[:d_var, k*d_var:(k+1)*d_var] = A_list[k]     # top block row
for k in range(p_var - 1):
    C[(k+1)*d_var:(k+2)*d_var, k*d_var:(k+1)*d_var] = np.eye(d_var)

eigs_companion = np.linalg.eigvals(C)
eig_mods       = np.abs(eigs_companion)

print("=== (b) Stationarity Check (Eigenvalue Method) ===")
print(f"  Companion matrix eigenvalue moduli: {np.round(eig_mods, 4)}")
is_stationary = (eig_mods < 1).all()
print(f"  All |λ| < 1 ? {'✓ Yes → Process is STATIONARY' if is_stationary else '✗ No → Process is NON-STATIONARY'}")

# ── (c) Estimate A1, A2, A3 from data via OLS (numpy only) ────────────────────
def fit_var_ols_numpy(X, p):
    """
    Fit VAR(p) by OLS using only numpy.
    Returns list of estimated [A1_hat, …, Ap_hat], each (d,d).
    """
    T_fit, d = X.shape
    # Regressor matrix Z: shape (T-p, p*d)
    # Row t: [X_{t-1}, X_{t-2}, …, X_{t-p}]  (each row-vector)
    Z = np.column_stack([X[p - k - 1: T_fit - k - 1] for k in range(p)])
    Y = X[p:]                               # (T-p, d)
    # OLS: B = (Z'Z)^{-1} Z'Y,  shape (p*d, d)
    B = np.linalg.lstsq(Z, Y, rcond=None)[0]
    # Reshape into list of (d,d) matrices: A_k = B[k*d:(k+1)*d].T
    A_hat = [B[k * d:(k + 1) * d].T for k in range(p)]
    return A_hat


A_hat_list = fit_var_ols_numpy(X_var3, p_var)

print("=== (c) Estimated coefficient matrices (OLS) ===")
for k, (A_true, A_hat) in enumerate(zip(A_list, A_hat_list)):
    print(f"  A{k+1} (true):\n{A_true}")
    print(f"  A{k+1} (estimated):\n{np.round(A_hat, 4)}")
    print(f"  Max |error|: {np.max(np.abs(A_true - A_hat)):.4f}")


---
## Q4 – Ridge Regression (Closed Form + sklearn Verification)

### Data generation
$X_k \stackrel{iid}{\sim} \mathcal{N}(\mathbf{0}, I_{20})$,  
four active indices $i_1,i_2,i_3,i_4 \in \{1,\ldots,20\}$ (without replacement),  
coefficients $a,b,c,d \sim \mathcal{N}(0,\,0.25)$,  
$Y_k = aX_k^{(i_1)} + bX_k^{(i_2)} + cX_k^{(i_3)} + dX_k^{(i_4)} + n_k$, $n_k \sim \mathcal{N}(0,0.01)$.

### Ridge closed form
$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = (X^\top X + \alpha I)^{-1} X^\top \mathbf{y}$$


In [ ]:
from sklearn.linear_model import Ridge

# ══════════════════════════════════════════════════════════════════
# PARAMETERS  (change freely)
# ══════════════════════════════════════════════════════════════════
N_Q4  = 100        # number of samples
P_Q4  = 20         # number of features
ALPHA_RIDGE = 20   # regularisation strength α
N_ACTIVE    = 4    # number of truly active features
# ══════════════════════════════════════════════════════════════════

# ── Data generation ────────────────────────────────────────────────────────────
X_q4   = np.random.multivariate_normal(np.zeros(P_Q4), np.eye(P_Q4), N_Q4)
idx4   = np.random.choice(P_Q4, N_ACTIVE, replace=False)     # active indices (0-based)
i1,i2,i3,i4 = idx4
print(f"Active indices (1-based): i1={i1+1}, i2={i2+1}, i3={i3+1}, i4={i4+1}")

# Coefficients ~ N(0, 0.25) → std = 0.5
a_r, b_r, c_r, d_r = np.random.normal(0, 0.5, N_ACTIVE)
print(f"True coefficients:  a={a_r:.4f}, b={b_r:.4f}, c={c_r:.4f}, d={d_r:.4f}")

noise_q4 = np.random.normal(0, 0.1, N_Q4)     # N(0, 0.01) → std = 0.1
y_q4     = (a_r * X_q4[:, i1] + b_r * X_q4[:, i2]
           + c_r * X_q4[:, i3] + d_r * X_q4[:, i4] + noise_q4)

# ── Ridge closed form ──────────────────────────────────────────────────────────
# β_ridge = (X'X + α I)^{-1} X'y
beta_ridge_cf = np.linalg.solve(
    X_q4.T @ X_q4 + ALPHA_RIDGE * np.eye(P_Q4),
    X_q4.T @ y_q4
)

# ── sklearn verification ───────────────────────────────────────────────────────
ridge_sk = Ridge(alpha=ALPHA_RIDGE, fit_intercept=False).fit(X_q4, y_q4)
beta_ridge_sk = ridge_sk.coef_

print(f"Ridge (α={ALPHA_RIDGE}) — closed form vs sklearn:")
print(f"  Max |difference| : {np.max(np.abs(beta_ridge_cf - beta_ridge_sk)):.2e}")
print(f"  Coefficients match: {np.allclose(beta_ridge_cf, beta_ridge_sk, atol=1e-6)}")

print("β_ridge (closed form):")
for j in range(P_Q4):
    marker = ' ← TRUE' if j in idx4 else ''
    print(f"  x{j+1:02d}: {beta_ridge_cf[j]:>8.4f}{marker}")

# ── Top-5 coefficients by |value| ─────────────────────────────────────────────
abs_order = np.argsort(np.abs(beta_ridge_cf))[::-1]
top5_idx  = abs_order[:5]
print(f"Top-5 indices by |β_ridge| (1-based): {[i+1 for i in top5_idx]}")
print(f"Their values: {beta_ridge_cf[top5_idx].round(4)}")
print(f"True active indices in top-5: {sorted(set(idx4) & set(top5_idx))}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['darkorange' if j in top5_idx else 'steelblue' for j in range(P_Q4)]
ax.bar(range(P_Q4), np.abs(beta_ridge_cf), color=colors, alpha=0.8)
for feat in idx4:
    ax.axvline(feat, color='r', lw=1.5, ls='--', alpha=0.7)
ax.set_xticks(range(P_Q4)); ax.set_xticklabels([f'x{j+1}' for j in range(P_Q4)], fontsize=8)
ax.set_title(f'|Ridge coefficients|  α={ALPHA_RIDGE}  — orange=top5, red-dashed=true active')
plt.tight_layout(); plt.show()


---
## Q5 – Regression Tree from Scratch (Squared Error Loss, Max Depth 2)

### Data
$X_k \stackrel{iid}{\sim} \mathcal{N}(\mathbf{0}, I_3)$,
$Y_k = |X_k^{(1)}| + |X_k^{(2)}| + |X_k^{(3)}|$

### Regression tree algorithm
At each node, find the split $(j^*, t^*)$ minimising weighted SSE:
$$\text{SSE}(j, t) = \sum_{x^{(j)} \le t}(y_i - \bar{y}_{\text{left}})^2 + \sum_{x^{(j)} > t}(y_i - \bar{y}_{\text{right}})^2$$
Predict the mean of the leaf's training points.

At each branching point we print: **which axis**, **threshold**, and **leaf constants** (predictions).


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS  (change freely)
# ══════════════════════════════════════════════════════════════════
N_Q5      = 1000
D_Q5      = 3           # feature dimension
MAX_DEPTH5 = 2          # max tree depth
# ══════════════════════════════════════════════════════════════════

np.random.seed(42)
X_q5 = np.random.multivariate_normal(np.zeros(D_Q5), np.eye(D_Q5), N_Q5)
y_q5 = np.sum(np.abs(X_q5), axis=1)    # Y = |x1| + |x2| + |x3|

# ── Squared-error helpers ──────────────────────────────────────────────────────
def sse(y):
    """Total squared error of y around its mean.  SSE = Σ(y_i - ȳ)²."""
    if len(y) == 0:
        return 0.0
    return np.sum((y - y.mean()) ** 2)


def best_split_reg(X, y):
    """
    Find (feature, threshold) minimising weighted SSE across both child nodes.
    Tries all midpoint thresholds between consecutive sorted unique values.
    Returns (best_feature_idx, best_threshold, best_weighted_sse).
    """
    n, n_feat = X.shape
    best_loss = np.inf
    bf, bt    = None, None

    for f in range(n_feat):
        vals = np.unique(X[:, f])
        thrs = (vals[:-1] + vals[1:]) / 2.0      # midpoints as candidate thresholds
        for th in thrs:
            mask = X[:, f] <= th
            if mask.sum() == 0 or (~mask).sum() == 0:
                continue
            loss = (sse(y[mask]) + sse(y[~mask])) / n
            if loss < best_loss:
                best_loss = loss; bf, bt = f, th

    return bf, bt, best_loss


def build_reg_tree(X, y, depth=0, max_depth=2, feat_names=None):
    """
    Recursively build a regression tree using squared error.
    Returns a dict node.  Prints split info at each branching point.
    """
    if feat_names is None:
        feat_names = [f'x{i+1}' for i in range(X.shape[1])]

    node = {
        'prediction': y.mean(),
        'n':          len(y),
        'sse':        sse(y),
        'depth':      depth,
    }

    # Stopping: max depth reached OR all y identical
    if depth >= max_depth or np.var(y) < 1e-12:
        node['is_leaf'] = True
        return node

    f, th, _ = best_split_reg(X, y)
    if f is None:
        node['is_leaf'] = True
        return node

    left_mask  = X[:, f] <= th
    right_mask = ~left_mask

    indent = "  " * depth
    print(f"{indent}Depth {depth}: Split on [{feat_names[f]}] ≤ {th:.4f}")
    print(f"{indent}  Left  n={left_mask.sum():4d}  mean(y)={y[left_mask].mean():.4f}")
    print(f"{indent}  Right n={right_mask.sum():4d}  mean(y)={y[right_mask].mean():.4f}")

    node['is_leaf']   = False
    node['feat']      = f
    node['feat_name'] = feat_names[f]
    node['threshold'] = th
    node['left']      = build_reg_tree(X[left_mask],  y[left_mask],  depth+1, max_depth, feat_names)
    node['right']     = build_reg_tree(X[right_mask], y[right_mask], depth+1, max_depth, feat_names)
    return node


def predict_reg_tree(node, X):
    """Predict using the fitted regression tree."""
    preds = np.empty(len(X))
    for i, x in enumerate(X):
        n = node
        while not n['is_leaf']:
            n = n['left'] if x[n['feat']] <= n['threshold'] else n['right']
        preds[i] = n['prediction']
    return preds


# ── Build from scratch ─────────────────────────────────────────────────────────
feat_names5 = [f'x{i+1}' for i in range(D_Q5)]

print("=== Regression Tree (from scratch) ===")
tree_q5 = build_reg_tree(X_q5, y_q5, max_depth=MAX_DEPTH5, feat_names=feat_names5)

y_pred_tree5 = predict_reg_tree(tree_q5, X_q5)
train_mse5   = mean_squared_error(y_q5, y_pred_tree5)
print(f"Training MSE (scratch): {train_mse5:.4f}")

# ── sklearn verification ───────────────────────────────────────────────────────
dt_q5 = DecisionTreeRegressor(max_depth=MAX_DEPTH5, random_state=42).fit(X_q5, y_q5)
y_pred_sk5  = dt_q5.predict(X_q5)
train_mse5_sk = mean_squared_error(y_q5, y_pred_sk5)

print(f"=== DecisionTreeRegressor (sklearn) ===")
print(export_text(dt_q5, feature_names=feat_names5))
print(f"Training MSE (sklearn): {train_mse5_sk:.4f}")
print(f"MSE match: {np.isclose(train_mse5, train_mse5_sk, atol=1e-4)}")

# ── Plot sklearn tree ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
plot_tree(dt_q5, feature_names=feat_names5, filled=True, ax=ax,
          rounded=True, fontsize=10)
ax.set_title(f'Regression Tree (depth≤{MAX_DEPTH5})  Training MSE={train_mse5_sk:.4f}')
plt.tight_layout(); plt.show()


---
## Q6 – Neural Network for $y_k = \log(e^{x_k^{(1)}} + e^{x_k^{(2)}})$

### Data
$x_i \stackrel{iid}{\sim} \text{Unif}[-2,2]^2$,  $y_i = \log(e^{x_i^{(1)}} + e^{x_i^{(2)}})$

### (a) 5-fold CV from scratch
Grid: $M_1, M_2 \in \{10, 20, 30\}$ (hidden layer sizes), activation $\in \{$logistic, tanh, relu$\}$  
→ $3 \times 3 \times 3 = 27$ estimators.  
**MLPRegressor** from sklearn fits the weights; **only the CV loop is implemented from scratch.**

### (b) Final fit + test error


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS  (change freely)
# ══════════════════════════════════════════════════════════════════
N_TRAIN_Q6 = 1000
N_TEST_Q6  = 1000
X_LOW, X_HIGH = -2.0, 2.0
M1_VALUES  = [10, 20, 30]          # first hidden layer sizes
M2_VALUES  = [10, 20, 30]          # second hidden layer sizes
ACTIVATIONS = ['logistic', 'tanh', 'relu']
K_FOLDS_Q6  = 5
MAX_ITER_Q6 = 1000                 # increase if convergence warning appears
# ══════════════════════════════════════════════════════════════════

np.random.seed(42)

# ── Data generation ────────────────────────────────────────────────────────────
X_q6_tr = np.random.uniform(X_LOW, X_HIGH, (N_TRAIN_Q6, 2))
y_q6_tr = np.log(np.exp(X_q6_tr[:, 0]) + np.exp(X_q6_tr[:, 1]))

print(f"Training data: X{X_q6_tr.shape}, y range [{y_q6_tr.min():.3f}, {y_q6_tr.max():.3f}]")

# ── (a) 5-fold CV from scratch ─────────────────────────────────────────────────
kf_q6  = KFold(n_splits=K_FOLDS_Q6, shuffle=True, random_state=0)
folds  = list(kf_q6.split(X_q6_tr))    # pre-compute fold indices

cv_results = {}
print(f"5-fold CV over 27 estimators (M1 × M2 × activation) ...")
print(f"{'M1':>4} {'M2':>4} {'Activation':>10}  {'CV MSE':>10}")
print("-" * 36)

for act in ACTIVATIONS:
    for M1 in M1_VALUES:
        for M2 in M2_VALUES:
            fold_mses = []
            for tr_idx, val_idx in folds:
                # Fit on training fold
                mlp = MLPRegressor(
                    hidden_layer_sizes=(M1, M2),   # two hidden layers
                    activation=act,
                    max_iter=MAX_ITER_Q6,
                    random_state=42
                ).fit(X_q6_tr[tr_idx], y_q6_tr[tr_idx])
                # Validate
                y_val_pred = mlp.predict(X_q6_tr[val_idx])
                fold_mses.append(mean_squared_error(y_q6_tr[val_idx], y_val_pred))

            cv_mse = np.mean(fold_mses)
            cv_results[(M1, M2, act)] = cv_mse
            print(f"{M1:>4} {M2:>4} {act:>10}  {cv_mse:>10.6f}")

# ── Best estimator ─────────────────────────────────────────────────────────────
best_key = min(cv_results, key=cv_results.get)
best_M1, best_M2, best_act = best_key
print(f"Best estimator: M1={best_M1}, M2={best_M2}, activation='{best_act}'")
print(f"Best CV MSE   : {cv_results[best_key]:.6f}")

# ── CV heatmap for best activation ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, act in zip(axes, ACTIVATIONS):
    mat = np.array([[cv_results[(M1, M2, act)] for M2 in M2_VALUES] for M1 in M1_VALUES])
    im  = ax.imshow(mat, cmap='YlOrRd_r', aspect='auto')
    ax.set_xticks(range(len(M2_VALUES))); ax.set_xticklabels(M2_VALUES)
    ax.set_yticks(range(len(M1_VALUES))); ax.set_yticklabels(M1_VALUES)
    ax.set_xlabel('M2'); ax.set_ylabel('M1'); ax.set_title(f'{act}')
    for i in range(len(M1_VALUES)):
        for j in range(len(M2_VALUES)):
            ax.text(j, i, f'{mat[i,j]:.4f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax)
plt.suptitle('5-Fold CV MSE Heatmap (each panel = one activation)', y=1.02)
plt.tight_layout(); plt.show()

# ── (b) Fit best model on full training data ───────────────────────────────────
best_mlp_q6 = MLPRegressor(
    hidden_layer_sizes=(best_M1, best_M2),
    activation=best_act,
    max_iter=MAX_ITER_Q6,
    random_state=42
).fit(X_q6_tr, y_q6_tr)

train_mse_q6 = mean_squared_error(y_q6_tr, best_mlp_q6.predict(X_q6_tr))
print(f"=== Final model: M1={best_M1}, M2={best_M2}, activation='{best_act}' ===")
print(f"  Training MSE  : {train_mse_q6:.6f}")
print(f"  Training RMSE : {np.sqrt(train_mse_q6):.6f}")

# ── Test data ──────────────────────────────────────────────────────────────────
X_q6_te = np.random.uniform(X_LOW, X_HIGH, (N_TEST_Q6, 2))
y_q6_te = np.log(np.exp(X_q6_te[:, 0]) + np.exp(X_q6_te[:, 1]))

test_mse_q6 = mean_squared_error(y_q6_te, best_mlp_q6.predict(X_q6_te))
print(f"  Test MSE      : {test_mse_q6:.6f}")
print(f"  Test RMSE     : {np.sqrt(test_mse_q6):.6f}")

# ── Residual plot ──────────────────────────────────────────────────────────────
y_te_pred = best_mlp_q6.predict(X_q6_te)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(y_q6_te, y_te_pred, s=10, alpha=0.5)
lo = min(y_q6_te.min(), y_te_pred.min()) - 0.1
hi = max(y_q6_te.max(), y_te_pred.max()) + 0.1
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='perfect prediction')
axes[0].set_xlabel('True y'); axes[0].set_ylabel('Predicted y')
axes[0].set_title(f'Test: True vs Predicted  (RMSE={np.sqrt(test_mse_q6):.4f})')
axes[0].legend()
axes[1].hist(y_q6_te - y_te_pred, bins=40, density=True, alpha=0.6, color='steelblue')
axes[1].set_title('Residual histogram (test)')
axes[1].set_xlabel('residual')
plt.tight_layout(); plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
print(f"{'='*55}")
print(f"  SUMMARY — Q6 Neural Network")
print(f"{'='*55}")
print(f"  Function:        y = log(e^x1 + e^x2)")
print(f"  Architecture:    2 hidden layers ({best_M1}, {best_M2}), {best_act}")
print(f"  Training MSE:    {train_mse_q6:.6f}")
print(f"  Test MSE:        {test_mse_q6:.6f}")
print(f"{'='*55}")


---
## Summary

| Q | Task | Key constraint | Method |
|---|------|---------------|--------|
| 1 | Poisson(λ) via Exp | No `chisquare()` | Cumsum of exponentials; manual χ² |
| 2 | MA(5) with uniform noise | numpy only | Loop MA filter; theoretical moments |
| 3 | VAR(3): generate, stationarity, estimate | numpy only | Cholesky noise; companion eigenvalues; OLS |
| 4 | Ridge regression | Closed form + sklearn verify | $(X^TX+αI)^{-1}X^Ty$ |
| 5 | Regression tree (depth 2) | From scratch | SSE split criterion; sklearn verify |
| 6 | 2-hidden-layer MLP, 27 configs | CV from scratch | Manual KFold loop; MLPRegressor weights |

**Generic design:** every function accepts user-supplied parameters.
Changing λ, MA order, VAR matrices, α, tree depth, or network architecture
requires editing only the **"PARAMETERS"** block — no structural changes needed.
